In [ ]:
import tensorflow as tf
import numpy as np
import os
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

✅ TensorFlow: 2.20.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ============================================================
# DESCARGAR BREAKHIS COMPLETO DESDE KAGGLE API
# ============================================================
import os

# Configurar credenciales Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"claudesistemas","key":"a777dbe64b88859696c89ae59a328930"}')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Descargar dataset
print("📥 Descargando BreakHis desde Kaggle...")
os.system('kaggle datasets download -d ambarish/breakhis -p /content/data --unzip')
print("✅ Descarga completa")

# Ver estructura
print("\n📁 Estructura:")
for root, dirs, files in os.walk('/content/data'):
    level = root.replace('/content/data', '').count(os.sep)
    if level < 4:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")

📥 Descargando BreakHis desde Kaggle...
✅ Descarga completa

📁 Estructura:
data/
  BreaKHis_v1/
    BreaKHis_v1/
      histology_slides/


In [ ]:
for root, dirs, files in os.walk('/content/data'):
    level = root.replace('/content/data', '').count(os.sep)
    if level < 7:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/ ({len(files)} archivos)")


data/ (1 archivos)
  BreaKHis_v1/ (0 archivos)
    BreaKHis_v1/ (0 archivos)
      histology_slides/ (0 archivos)
        breast/ (2 archivos)
          benign/ (6 archivos)
            SOB/ (0 archivos)
          malignant/ (6 archivos)
            SOB/ (0 archivos)


In [ ]:
benign = 0
malignant = 0

for root, dirs, files in os.walk('/content/data/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign'):
    benign += len([f for f in files if f.lower().endswith('.png')])

for root, dirs, files in os.walk('/content/data/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/malignant'):
    malignant += len([f for f in files if f.lower().endswith('.png')])

print(f"✅ Benignas:   {benign}")
print(f"✅ Malignas:   {malignant}")
print(f"✅ Total:      {benign + malignant}")

✅ Benignas:   2480
✅ Malignas:   5429
✅ Total:      7909


In [ ]:
# ============================================================
# ENTRENAMIENTO CÁNCER DE MAMA — EfficientNetB0
# Dataset: BreaKHis completo 7,909 imágenes
# Autor: Danner Jamanca
# ============================================================

import tensorflow as tf
import numpy as np
import os
import shutil
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

# ============================================================
# 1. ORGANIZAR IMÁGENES
# ============================================================
BASE = '/content/data/BreaKHis_v1/BreaKHis_v1/histology_slides/breast'
OUTPUT = '/content/dataset_procesado'

# Limpiar si ya existe
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)

def recolectar_imagenes(clase_path):
    imagenes = []
    for root, dirs, files in os.walk(clase_path):
        for f in files:
            if f.lower().endswith('.png'):
                imagenes.append(os.path.join(root, f))
    return imagenes

benignas = recolectar_imagenes(f'{BASE}/benign')
malignas = recolectar_imagenes(f'{BASE}/malignant')
print(f"✅ Benignas: {len(benignas)} | Malignas: {len(malignas)}")

# Split 70/15/15
b_train, b_temp = train_test_split(benignas, test_size=0.30, random_state=42)
b_val, b_test   = train_test_split(b_temp,   test_size=0.50, random_state=42)
m_train, m_temp = train_test_split(malignas, test_size=0.30, random_state=42)
m_val, m_test   = train_test_split(m_temp,   test_size=0.50, random_state=42)

splits = {
    'train':      {'normal': b_train, 'anormal': m_train},
    'validation': {'normal': b_val,   'anormal': m_val},
    'test':       {'normal': b_test,  'anormal': m_test},
}

for split, clases in splits.items():
    for clase, archivos in clases.items():
        dest = f'{OUTPUT}/{split}/{clase}'
        os.makedirs(dest, exist_ok=True)
        for src in archivos:
            shutil.copy2(src, dest)

print("\n✅ Dataset organizado:")
for split in ['train', 'validation', 'test']:
    for clase in ['normal', 'anormal']:
        n = len(os.listdir(f'{OUTPUT}/{split}/{clase}'))
        print(f"   {split}/{clase}: {n}")

# ============================================================
# 2. GENERADORES — preprocess_input correcto para EfficientNet
# ============================================================
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    f'{OUTPUT}/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED
)
val_gen = val_test_datagen.flow_from_directory(
    f'{OUTPUT}/validation', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED
)
test_gen = val_test_datagen.flow_from_directory(
    f'{OUTPUT}/test', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

print(f"\n✅ Clases: {train_gen.class_indices}")
print(f"✅ Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")

# ============================================================
# 3. CLASS WEIGHTS
# ============================================================
labels = train_gen.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))
print(f"\n✅ Class weights: {class_weight_dict}")

# ============================================================
# 4. MODELO
# ============================================================
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
print(f"\n✅ Modelo creado: {model.count_params():,} parámetros")

# ============================================================
# 5. FASE 1 — Solo cabeza
# ============================================================
print("\n🚀 FASE 1 — Entrenando cabeza...")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_f1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

history1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=20, callbacks=callbacks_f1,
    class_weight=class_weight_dict, verbose=1
)

# ============================================================
# 6. FASE 2 — Fine tuning
# ============================================================
print("\n🚀 FASE 2 — Fine tuning últimas 30 capas...")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_f2 = [
    ModelCheckpoint('mejor_modelo_mama.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1),
]

history2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=50, callbacks=callbacks_f2,
    class_weight=class_weight_dict, verbose=1
)

# ============================================================
# 7. EVALUACIÓN
# ============================================================
print("\n📊 Evaluando en test set...")
test_gen.reset()
results = model.evaluate(test_gen, verbose=1)
accuracy = results[1] * 100

print(f"""
╔════════════════════════════════════════╗
║  PRECISIÓN FINAL: {accuracy:.2f}%{' ' * (18 - len(f'{accuracy:.2f}'))}║
╚════════════════════════════════════════╝
""")

# ============================================================
# 8. GUARDAR Y DESCARGAR
# ============================================================
model.save('modelo_mama.keras')

from google.colab import files
files.download('mejor_modelo_mama.keras')
print("📥 Descargando mejor modelo...")

✅ TensorFlow: 2.20.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Benignas: 2480 | Malignas: 5429

✅ Dataset organizado:
   train/normal: 1736
   train/anormal: 3800
   validation/normal: 372
   validation/anormal: 814
   test/normal: 372
   test/anormal: 815
Found 5536 images belonging to 2 classes.
Found 1186 images belonging to 2 classes.
Found 1187 images belonging to 2 classes.

✅ Clases: {'anormal': 0, 'normal': 1}
✅ Train: 5536 | Val: 1186 | Test: 1187

✅ Class weights: {0: np.float64(0.728421052631579), 1: np.float64(1.5944700460829493)}

✅ Modelo creado: 4,415,652 parámetros

🚀 FASE 1 — Entrenando cabeza...
Epoch 1/20
173/173 ━━━━━━━━━━━━━━━━━━━━ 199s 1s/step - accuracy: 0.7823 - loss: 0.4871 - val_accuracy: 0.7690 - val_loss: 0.4976 - learning_rate: 0.0010
Epoch 2/20
173/173 ━━━━━━━━━━━━━━━━━━━━ 150s 869ms/step - accuracy: 0.8289 - loss: 0.4010 - val_accuracy: 0.8128 - val_loss: 0.4161 - learning_rate: 0.0010
Epoch 3/20
173/173 ━━━━━━━━━━━━━━━━━

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descargando mejor modelo...
